<a href="https://colab.research.google.com/github/kb0417/french-ewe-translation-transcription/blob/main/notebooks/06_finetuning_nllb_french_ewe.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 06 — Fine-tuning de NLLB pour la traduction français → éwé

Dans le notebook précédent, nous avons testé NLLB-200 distilled 600M sans entraînement supplémentaire.

Les résultats qualitatifs étaient meilleurs que ceux des modèles LSTM, mais les scores BLEU restaient faibles sur notre jeu de test.

Dans ce notebook, nous allons adapter NLLB à notre corpus parallèle français-éwé grâce à un fine-tuning.

L’objectif est d’améliorer la traduction français → éwé en utilisant les paires de phrases de notre dataset.

In [15]:
# On installe les bibliothèques nécessaires.
# transformers permet de charger et fine-tuner NLLB.
# datasets facilite la préparation des données.
# evaluate et sacrebleu servent pour l'évaluation.
# peft permet de faire un fine-tuning léger avec LoRA, plus adapté à Google Colab.

!pip install -U transformers datasets evaluate sacrebleu sentencepiece accelerate peft torchao -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 13.8 MB/s eta 0:00:00


In [1]:
import os
import torch
import pandas as pd
import numpy as np

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)

from peft import LoraConfig, get_peft_model, TaskType
import evaluate

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Appareil utilisé :", device)

if device == "cuda":
    print("GPU :", torch.cuda.get_device_name(0))
else:
    print("Attention : sans GPU, le fine-tuning sera très lent.")

Appareil utilisé : cuda
GPU : Tesla T4


In [3]:
from google.colab import drive
drive.mount('/content/drive')

project_dir = "/content/drive/MyDrive/french_ewe_project"

data_dir = os.path.join(project_dir, "data/processed")
models_dir = os.path.join(project_dir, "models")
reports_dir = os.path.join(project_dir, "reports")

os.makedirs(models_dir, exist_ok=True)
os.makedirs(reports_dir, exist_ok=True)

print("Dossier données :", data_dir)
print("Dossier modèles :", models_dir)
print("Dossier rapports :", reports_dir)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dossier données : /content/drive/MyDrive/french_ewe_project/data/processed
Dossier modèles : /content/drive/MyDrive/french_ewe_project/models
Dossier rapports : /content/drive/MyDrive/french_ewe_project/reports


In [4]:
from google.colab import drive
drive.mount('/content/drive')

project_dir = "/content/drive/MyDrive/french_ewe_project"

data_dir = os.path.join(project_dir, "data/processed")
models_dir = os.path.join(project_dir, "models")
reports_dir = os.path.join(project_dir, "reports")

os.makedirs(models_dir, exist_ok=True)
os.makedirs(reports_dir, exist_ok=True)

print("Dossier données :", data_dir)
print("Dossier modèles :", models_dir)
print("Dossier rapports :", reports_dir)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dossier données : /content/drive/MyDrive/french_ewe_project/data/processed
Dossier modèles : /content/drive/MyDrive/french_ewe_project/models
Dossier rapports : /content/drive/MyDrive/french_ewe_project/reports


In [5]:
train_df = pd.read_csv(os.path.join(data_dir, "train.csv"))
valid_df = pd.read_csv(os.path.join(data_dir, "valid.csv"))
test_df = pd.read_csv(os.path.join(data_dir, "test.csv"))

print("Train :", train_df.shape)
print("Validation :", valid_df.shape)
print("Test :", test_df.shape)

train_df.head()

Train : (18820, 6)
Validation : (2352, 6)
Test : (2353, 6)


,french,ewe,source,type,french_length,ewe_length
0,"""C'est moins lourd pour les enfants, pour le s...",ele hodzoe na ɖeviwo kple ame bubu geɖewo,https://ellecitoyenne.com/,"Blog, News",14,8
1,"""Il ne nous reste que nous, si nous voulons un...",mía ŋutɔwo dzi ɖeɖe ko wole be míaka ɖo ne m...,https://ellecitoyenne.com/,"Blog, News",15,13
2,"""Pendant quelques années, Histoire et History ...",tsakakatsaka ɖeke magava eme le akɔdada le kam...,https://ellecitoyenne.com/,"Blog, News",47,14
3,"""Ainsi, à travers une lecture littérale de la ...",Aleae to bibla xexle me la miedoa dzesi be duk...,https://ellecitoyenne.com/,"Blog, News",21,21
4,"""Le prof d'EPS qui donne des cours de langue ?...",meli ʋliʋlim xena fetu sɔsɔe xɔxɔ le dɔ ƒomevi...,https://ellecitoyenne.com/,"Blog, News",45,11


In [6]:
# Pour un premier essai, on ne prend pas tout le dataset.
# Cela permet de vérifier que le pipeline fonctionne sans attendre plusieurs heures.
#
# Si tout fonctionne bien, on pourra augmenter ces valeurs.

TRAIN_SAMPLE_SIZE = 3000
VALID_SAMPLE_SIZE = 300
TEST_SAMPLE_SIZE = 200

train_small = train_df.sample(TRAIN_SAMPLE_SIZE, random_state=42).reset_index(drop=True)
valid_small = valid_df.sample(VALID_SAMPLE_SIZE, random_state=42).reset_index(drop=True)
test_small = test_df.sample(TEST_SAMPLE_SIZE, random_state=42).reset_index(drop=True)

print("Train small :", train_small.shape)
print("Valid small :", valid_small.shape)
print("Test small :", test_small.shape)

Train small : (3000, 6)
Valid small : (300, 6)
Test small : (200, 6)


In [7]:
# Pour le fine-tuning français → éwé :
# - source = phrase française
# - target = phrase éwé

train_small = train_small[["french", "ewe"]].dropna()
valid_small = valid_small[["french", "ewe"]].dropna()
test_small = test_small[["french", "ewe"]].dropna()

train_small.head()

,french,ewe
0,D'autres ne peuvent pas s'acheter des tampons ...,bubuwo mateŋjaƒle tampɔ̃ o esiatae wole aƒelme...
1,"En lisant la fin, on suppose que « John Ziguéh...",nuwuwua ɖee fia be « John Ziguéhi » ɖo eƒe ŋug...
2,Il est 18h30 lorsque je descends enfin du boulot,eƒo ga adẽ kple afã eye medo le dɔa me
3,Le discours des grandes figures africaines du ...,Afrika ame ŋkuta gã siawo abe nye Chimamand...
4,"Amoureuse, elle a cédé à ses avances, à ses ba...",lɔlɔ̃tɔe la elɔ̃ ɖe eƒe asiliameŋu kple dzidzi...


In [24]:
train_dataset = Dataset.from_pandas(train_small)
valid_dataset = Dataset.from_pandas(valid_small)
test_dataset = Dataset.from_pandas(test_small)

train_dataset

Dataset({
    features: ['french', 'ewe'],
    num_rows: 3000
})

In [25]:
model_name = "facebook/nllb-200-distilled-600M"

tokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("NLLB chargé.")

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

NLLB chargé.


In [26]:
FRENCH_CODE = "fra_Latn"
EWE_CODE = "ewe_Latn"

tokenizer.src_lang = FRENCH_CODE
base_model.generation_config.forced_bos_token_id = tokenizer.convert_tokens_to_ids(EWE_CODE)

In [27]:
# LoRA permet d'adapter un grand modèle sans entraîner tous ses paramètres.
# Cela réduit fortement la mémoire nécessaire et accélère l'entraînement.

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_2_SEQ_LM
)

model = get_peft_model(base_model, lora_config)

model.print_trainable_parameters()

trainable params: 1,179,648 || all params: 616,253,440 || trainable%: 0.1914


In [28]:
# On applique aussi la configuration de génération au modèle LoRA.
# Cela évite l'erreur pendant l'évaluation avec Seq2SeqTrainer.

model.generation_config.forced_bos_token_id = tokenizer.convert_tokens_to_ids(EWE_CODE)

In [29]:
MAX_INPUT_LENGTH = 128
MAX_TARGET_LENGTH = 128

In [30]:
def preprocess_function(examples):
    # On indique à NLLB que la langue source est le français.
    tokenizer.src_lang = FRENCH_CODE

    # Phrases sources : français
    inputs = examples["french"]

    # Phrases cibles : éwé
    targets = examples["ewe"]

    # Tokenisation des phrases sources.
    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT_LENGTH,
        truncation=True
    )

    # Tokenisation des phrases cibles.
    labels = tokenizer(
        text_target=targets,
        max_length=MAX_TARGET_LENGTH,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

In [31]:
tokenized_train = train_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=train_dataset.column_names
)

tokenized_valid = valid_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=valid_dataset.column_names
)

tokenized_test = test_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=test_dataset.column_names
)

tokenized_train

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/300 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 3000
})

In [32]:
# Le data collator prépare les batchs pendant l'entraînement.
# Il applique le padding automatiquement selon la taille des phrases dans chaque batch.

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

In [33]:
sacrebleu_metric = evaluate.load("sacrebleu")

In [34]:
def postprocess_text(preds, labels):
    preds = [pred.strip() for pred in preds]
    labels = [[label.strip()] for label in labels]
    return preds, labels


def compute_metrics(eval_preds):
    preds, labels = eval_preds

    # Certains modèles retournent un tuple.
    if isinstance(preds, tuple):
        preds = preds[0]

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    # Les labels contiennent parfois -100 pour ignorer certaines positions.
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)

    result = sacrebleu_metric.compute(
        predictions=decoded_preds,
        references=decoded_labels
    )

    return {"bleu": result["score"]}

In [35]:
output_dir = os.path.join(models_dir, "nllb_fr_to_ewe_lora")

In [36]:
training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,

    # On commence petit pour éviter les crashs mémoire.
    num_train_epochs=3,
    learning_rate=2e-4,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,

    predict_with_generate=True,
    generation_max_length=128,
    generation_num_beams=4,

    # Dans ta version de transformers, il faut utiliser eval_strategy
    # et non evaluation_strategy.
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,

    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="bleu",
    greater_is_better=True,

    fp16=torch.cuda.is_available(),

    report_to="none"
)

In [37]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,

    train_dataset=tokenized_train,
    eval_dataset=tokenized_valid,

    processing_class=tokenizer,
    data_collator=data_collator,

    compute_metrics=compute_metrics
)

In [38]:
# On lance le fine-tuning.
# Avec LoRA et un sous-ensemble de données, cette étape doit rester raisonnable sur Colab.

trainer.train()

Epoch,Training Loss,Validation Loss,Bleu
1,40.200464,4.156519,4.447571
2,31.897307,3.811957,4.854116
3,30.633347,3.770900,4.904719


TrainOutput(global_step=564, training_loss=37.73613987915905, metrics={'train_runtime': 1084.6508, 'train_samples_per_second': 8.298, 'train_steps_per_second': 0.52, 'total_flos': 739419946008576.0, 'train_loss': 37.73613987915905, 'epoch': 3.0})

In [39]:
test_results = trainer.evaluate(tokenized_test)

test_results

Training Loss,Validation Loss,Epoch,Bleu
30.633347,3.559422,3,5.963365


{'eval_loss': 3.559422492980957, 'eval_bleu': 5.963364815107571}

In [40]:
# On sauvegarde l'adaptateur LoRA fine-tuné.
# Le modèle complet NLLB n'est pas recopié entièrement : seuls les poids LoRA sont sauvegardés.

finetuned_model_dir = os.path.join(models_dir, "nllb_fr_to_ewe_lora_final")

trainer.save_model(finetuned_model_dir)
tokenizer.save_pretrained(finetuned_model_dir)

print("Modèle fine-tuné sauvegardé ici :", finetuned_model_dir)

Modèle fine-tuné sauvegardé ici : /content/drive/MyDrive/french_ewe_project/models/nllb_fr_to_ewe_lora_final


In [41]:
def translate_finetuned_nllb_fr_to_ewe(text, max_length=128):
    tokenizer.src_lang = FRENCH_CODE

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=max_length
    ).to(model.device)

    generated_tokens = model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.convert_tokens_to_ids(EWE_CODE),
        max_length=max_length,
        num_beams=4
    )

    translation = tokenizer.batch_decode(
        generated_tokens,
        skip_special_tokens=True
    )[0]

    return translation

In [42]:
phrases_test = [
    "Bonjour",
    "Comment tu vas ?",
    "Je suis malade",
    "Je veux boire de l'eau",
    "Merci beaucoup",
    "Je vais à la maison"
]

for phrase in phrases_test:
    traduction = translate_finetuned_nllb_fr_to_ewe(phrase)

    print("Français :", phrase)
    print("Éwé fine-tuné :", traduction)
    print("-" * 80)

Français : Bonjour
Éwé fine-tuné : gbedoname
--------------------------------------------------------------------------------
Français : Comment tu vas ?
Éwé fine-tuné : Nu ka wɔm nèle?
--------------------------------------------------------------------------------
Français : Je suis malade
Éwé fine-tuné : mele dɔ lém
--------------------------------------------------------------------------------
Français : Je veux boire de l'eau
Éwé fine-tuné : medina be mano tsi
--------------------------------------------------------------------------------
Français : Merci beaucoup
Éwé fine-tuné : meda akpe geɖe
--------------------------------------------------------------------------------
Français : Je vais à la maison
Éwé fine-tuné : meyina aƒe me
--------------------------------------------------------------------------------


In [43]:
for i in range(5):
    french_sentence = test_small.iloc[i]["french"]
    true_ewe = test_small.iloc[i]["ewe"]

    predicted_ewe = translate_finetuned_nllb_fr_to_ewe(french_sentence)

    print("Français original :", french_sentence)
    print("Éwé attendu       :", true_ewe)
    print("Éwé fine-tuné     :", predicted_ewe)
    print("-" * 100)

Français original : Quand le lézard fut bien cuit, Mariko, se mit à le manger.
Éwé attendu       : Esi doglo bi nyuie la, Mariko dze eɖuɖu.
Éwé fine-tuné     : Esi woɖa lãa nyuie la, Mariko dze edzi ɖui.
----------------------------------------------------------------------------------------------------
Français original : Je l'ai porté à  cette convention parce que c'était un évènement islamique,
Éwé attendu       : Medoe yi takpekpe sia elabe enye moslemtɔwo ƒe ɖoɖo
Éwé fine-tuné     : metsɔe yi takpekpe sia me le esi wònye Islamtɔwo ƒe wɔna aɖe ta
----------------------------------------------------------------------------------------------------
Français original :  Je suis revenu dans la salle de séjour, où tu avais l’air de m’attendre
Éwé attendu       : metrɔ va asadzi afi si edze kɔtɛ be enɔ te nɔm kpɔm le lae
Éwé fine-tuné     : megatrɔ yi xɔ si me nèle lalam le la me
----------------------------------------------------------------------------------------------------
Français 

In [44]:
results_finetuning = pd.DataFrame({
    "model": ["NLLB LoRA fine-tuned"],
    "direction": ["Français → Éwé"],
    "train_samples": [TRAIN_SAMPLE_SIZE],
    "valid_samples": [VALID_SAMPLE_SIZE],
    "test_samples": [TEST_SAMPLE_SIZE],
    "test_bleu": [test_results["eval_bleu"]]
})

results_path = os.path.join(reports_dir, "nllb_finetuning_fr_to_ewe_results.csv")
results_finetuning.to_csv(results_path, index=False)

results_finetuning

,model,direction,train_samples,valid_samples,test_samples,test_bleu
0,NLLB LoRA fine-tuned,Français → Éwé,3000,300,200,5.963365


## Conclusion

Dans ce notebook, nous avons réalisé un fine-tuning léger de NLLB pour la traduction français → éwé.

Pour rendre l'entraînement possible sur Google Colab, nous avons utilisé LoRA, une méthode de fine-tuning paramètre-efficace. Au lieu de modifier tous les paramètres du modèle NLLB, LoRA ajoute et entraîne seulement un petit nombre de paramètres.

Cette approche permet d'adapter NLLB à notre corpus français-éwé tout en limitant la mémoire nécessaire.

Le modèle fine-tuné est ensuite évalué avec BLEU sur un échantillon du jeu de test, puis testé qualitativement sur quelques phrases simples et quelques phrases issues du dataset.

Cette étape constitue une amélioration importante par rapport aux modèles LSTM entraînés from scratch, car elle combine les connaissances multilingues de NLLB avec notre corpus spécialisé français-éwé.

## Résultats obtenus

Le fine-tuning LoRA de NLLB a été réalisé sur un sous-ensemble du corpus :

- 3000 phrases d'entraînement ;
- 300 phrases de validation ;
- 200 phrases de test.

Les résultats obtenus sur le jeu de test sont :

- Loss : 3.5594
- BLEU : 5.9634

Ce score BLEU est supérieur à celui obtenu avec NLLB pré-entraîné sans fine-tuning sur le même sens de traduction. Cela montre que l’adaptation du modèle au corpus français-éwé améliore la qualité des traductions.

## Analyse qualitative

Les traductions générées après fine-tuning sont plus cohérentes que celles produites par les modèles LSTM.  
Par exemple, le modèle traduit correctement plusieurs phrases simples comme “Je suis malade”, “Merci beaucoup” ou “Je vais à la maison”.

Sur les phrases du jeu de test, les sorties générées ne correspondent pas toujours exactement aux références, mais elles conservent souvent une partie importante du sens général.

Cela confirme l’intérêt d’utiliser un modèle multilingue pré-entraîné comme NLLB, puis de l’adapter au corpus spécifique à l’aide d’un fine-tuning léger avec LoRA.